## P25_37d_book — multi-parameter probe exports (book figures)

Book-figure version of the P25_37d video notebook. Picks **5 parameters** (plain8, screwdriver image), and for each one:

| what | where | naming |
|---|---|---|
| 1000-class prob plot at each probe value | `p37_param_k/probs/` | `trained.svg`, `00_-2.5.svg` … `10_+2.5.svg` |
| θ = / P(class) = equation at each probe value | `p37_param_k/equations/` | same naming as probs |
| P(correct) vs θ curve, no dot | `p37_param_k/curve/` | `curve.svg` |
| same curve, dot at the current probe value | `p37_param_k/curve/` | `curve_dot_trained.svg`, `curve_dot_00_-2.5.svg` … |
| metadata + probe results | `p37_param_k/` | `summary.json` |

Changes vs the video version: no activation caches, no draw-in / dot-moving frames, no "other curves" re-rendering; everything is
transparent-background SVG; the parameter is probed at `PROBE_VALS` = -2.5, -2.0, …, 2.5 (11 values). The final-layer parameter
is chosen by max |grad| **restricted to the weights feeding output unit 951 (lemon)**.

Conventions: grad ranking is computed once at the trained weights; each parameter is restored to its trained value after its probe.

In [ ]:
BLUE="#2ca3dd"
YELLOW="#ffd35a"
RED='#ec2027'
CHILL_BROWN="#948979"

In [ ]:
import json, time
from pathlib import Path
from tqdm import tqdm
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.figure import Figure   # OO figures for batch rendering: never touch pyplot/the notebook backend
from torchvision.models.resnet import ResNet, BasicBlock
import torchvision.datasets as dsets, torchvision.transforms as T

plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'serif'
plt.rcParams['svg.fonttype'] = 'none'   # keep text as editable text in the SVGs (set to 'path' to outline glyphs instead)

RUNS     = Path("/home/stephen/Stephencwelch Dropbox/welch_labs/resnet/hackin/aug_17_run")   # output dir of train_depth_sweep.py
DATA     = Path("/home/stephen/imagenet")
OUT_ROOT = Path("/home/stephen/Stephencwelch Dropbox/welch_labs/ai_book_vol_2/3_resnets/graphics")   # p37_param_k folders land here
device   = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_DEPTHS = {"plain8": 8, "plain14": 14, "plain20": 20, "plain26": 26,
                "plain34": 34, "plain56": 56, "plain74": 74, "resnet74": 74}
print(torch.__version__, device)

### Model + data

In [ ]:
class PlainBasicBlock(BasicBlock):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.downsample = None

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        return out

LAYER_CFG = {8: [1,1,1,1], 14: [2,1,1,2], 20: [2,2,3,2], 26: [3,3,3,3],
             34: [4,4,4,4], 56: [3,4,17,3], 74: [3,4,26,3]}

def make_net(depth_target, use_skip, num_classes=1000):
    block = BasicBlock if use_skip else PlainBasicBlock
    model = ResNet(block, LAYER_CFG[depth_target], num_classes=num_classes)
    if depth_target == 8:
        model.layer4 = nn.Identity()
        model.fc = nn.Linear(256, num_classes)
    return model

def load_model(name, step=None):
    d = RUNS / name
    path = d / "final.pt" if step is None else d / f"ckpt_step{step:06d}.pt"
    model = make_net(MODEL_DEPTHS[name], use_skip=name.startswith("resnet"))
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()

def weighted_layers(model):
    """(name, module) for convs + fc in forward order, excluding 1x1 shortcuts."""
    return [(n, m) for n, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear)) and "downsample" not in n]

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
val_ds = dsets.ImageFolder(DATA / "ILSVRC/Data/CLS-LOC/val",
    T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)]))

wnid_words = {}
mapping = DATA / "LOC_synset_mapping.txt"
if mapping.exists():
    for line in open(mapping):
        wnid, words = line.strip().split(" ", 1)
        wnid_words[wnid] = words.split(",")[0]

def class_name(idx):
    wnid = val_ds.classes[idx]
    return wnid_words.get(wnid, wnid)

def denorm(x):  # CHW tensor -> HWC numpy in [0,1] for imshow
    img = x.cpu() * torch.tensor(STD).view(3,1,1) + torch.tensor(MEAN).view(3,1,1)
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

def load_image(idx):
    x, y = val_ds[idx]
    return x.unsqueeze(0).to(device), torch.tensor([y], device=device)

print(len(val_ds), "val images,", len(val_ds.classes), "classes")

### Sweep helpers

In [ ]:
crit = nn.CrossEntropyLoss()

@torch.no_grad()
def all_probs(model, x):
    """softmax over the 1000 classes, as a numpy vector."""
    return F.softmax(model(x), dim=1)[0].cpu().numpy()

@torch.no_grad()
def correct_ans_conf(model, x, y):
    return F.softmax(model(x), dim=1)[0, y.item()].item()

def layer_grad(model, layer, x, y):
    """dL/dw for one layer at the current weights, single backward, fp32."""
    model.zero_grad(set_to_none=True)
    crit(model(x), y).backward()
    g = layer.weight.grad.detach().flatten().clone()
    model.zero_grad(set_to_none=True)
    return g

@torch.no_grad()
def sweep_weight(model, layer, flat_idx, x, y, span, n):
    """Set one weight to each value in linspace(-span, span, n), record P(correct), restore. Returns (vals, res, w0)."""
    w = layer.weight.view(-1)
    w0 = w[flat_idx].item()
    vals = np.linspace(-span, span, n)
    res = np.empty(n)
    for i, v in enumerate(vals):
        w[flat_idx] = v
        res[i] = correct_ans_conf(model, x, y)
    w[flat_idx] = w0
    return vals, res, w0

### Rendering helpers (transparent-background SVG)

In [ ]:
# --- figure geometry (shared by every export so nothing jitters between figures) ---
# NOTE: the render_* helpers build matplotlib.figure.Figure objects directly. These are never registered with
# pyplot or the notebook backend, so many exports don't accumulate in RAM.
PROBS_FIGSIZE  = (2.0, 8)
PROBS_AXES     = [0.25, 0.03, 0.7, 0.9]
EQ_FIGSIZE     = (6, 2)
EQ_ANCHOR_X    = 0.60            # fig-fraction x of the '=' sign
EQ_FONTSIZE    = 28
CURVE_FIGSIZE  = (6, 6)
CURVE_YLIM     = (-0.05, 1.05)

def save_svg(fig, path, **kw):
    fig.savefig(path, format='svg', transparent=True, **kw)
    fig.clear()

def style_ax(ax, color=CHILL_BROWN, tick_fontsize=12, keep_spines=False):
    if keep_spines: ax.spines[:].set_color(color)
    else: ax.spines[:].set_visible(False)
    ax.tick_params(colors=color, which='both', labelsize=tick_fontsize)
    ax.xaxis.label.set_color(color)
    ax.yaxis.label.set_color(color)
    ax.title.set_color(color)

def theta_latex(coord, layer_num):
    """(784, 103), 8  ->  $\\theta_{(784,\\,103)}^{(8)}$"""
    inner = r',\,'.join(str(int(c)) for c in coord)
    return r'$\theta_{(' + inner + r')}^{(' + str(layer_num) + r')}$'

def render_probs(probs, y_true, path):
    """Vertical 1000-class softmax plot, dot on the true class."""
    fig = Figure(figsize=PROBS_FIGSIZE)
    ax = fig.add_axes(PROBS_AXES)
    ax.set_facecolor('none')
    ax.plot(probs, np.arange(len(probs)), c=BLUE, linewidth=1.65)
    ax.scatter(probs[y_true], y_true, c=YELLOW, s=30, zorder=100)
    ax.set_ylim(len(probs), 0)          # fixed ylim so the figures line up
    ax.xaxis.tick_top()
    ax.set_xlim(left=0)                  # right edge autoscales
    style_ax(ax)
    save_svg(fig, path)

def render_equation(theta_label, theta_val, p_label, p_val, theta_color, path):
    """Two-line 'theta = v' / 'P(class) = p' figure."""
    fig = Figure(figsize=EQ_FIGSIZE)
    for yy, label, val, col in zip([0.68, 0.32], [theta_label, p_label], [theta_val, p_val], [theta_color, YELLOW]):
        fig.text(EQ_ANCHOR_X - 0.05, yy, label,          ha='right',  va='center', fontsize=EQ_FONTSIZE, color=col)
        fig.text(EQ_ANCHOR_X,        yy, '$=$',          ha='center', va='center', fontsize=EQ_FONTSIZE, color=col)
        fig.text(EQ_ANCHOR_X + 0.05, yy, f'${val:.3f}$', ha='left',   va='center', fontsize=EQ_FONTSIZE, color=col)
    save_svg(fig, path)

def render_curve(vals, res, color, path, dot=None, dot_size=125):
    """P(correct) vs theta. dot=(x, y) -> marker at (x, y) on top of the curve; dot=None -> no marker."""
    fig = Figure(figsize=CURVE_FIGSIZE)
    ax = fig.add_subplot(111)
    ax.set_facecolor('none')
    ax.plot(vals, res, linewidth=3, c=color)
    if dot is not None: ax.scatter(*dot, s=dot_size, c=color, zorder=10)
    ax.grid(True, color=CHILL_BROWN, alpha=0.3, linewidth=0.8)
    ax.set_xlim(-SPAN, SPAN)   # ylim left adaptive; set ax.set_ylim(*CURVE_YLIM) for a fixed [0,1] axis
    style_ax(ax)
    save_svg(fig, path, bbox_inches='tight')

## Config

`N_POINTS = 201` gives a 0.025 step, so every value in `PROBE_VALS` lies exactly on the curve grid.

In [ ]:
MODEL_NAME = 'plain8'
IMG_IDX    = 39209                 # screwdriver
SPAN       = 2.5
N_POINTS   = 201                   # resolution of the P(correct) vs theta curve
PROBE_VALS = np.arange(-SPAN, SPAN + 1e-9, 0.5)   # -2.5, -2.0, ..., 2.5  (11 values)
TOP_N      = 256                   # how many |grad|-ranked weights to consider per layer
P_LABEL    = r'$P(\mathrm{Screw\ Driver})$'
LEMON      = 951                   # ImageNet class index for lemon

# li indexes weighted_layers(model) (convs + fc, forward order); grad_index ranks by |dL/dw| at the trained weights.
# out_idx (optional) restricts the ranking to weights feeding that output unit / channel (row 0 of the weight tensor),
# e.g. out_idx=951 on the fc layer -> a weight directly hooked up to the "lemon" logit.
SWEEPS = [
    dict(li=-1, grad_index=0, color='m',       out_idx=LEMON),
    dict(li=-3, grad_index=0, color='#eb8423'),
    dict(li=3,  grad_index=1, color='#419c52'),
    dict(li=2,  grad_index=7, color='#ed5e78'),
    dict(li=0,  grad_index=0, color='#d73b2f'),
]
SWEEPS_TO_RUN = [1, 2, 3, 4, 5]    # 1-indexed; run a subset if you want
assert len(PROBE_VALS) == 11, PROBE_VALS

In [ ]:
model = load_model(MODEL_NAME)
x, y  = load_image(IMG_IDX)
y_true = y.item()
base_conf = correct_ans_conf(model, x, y)
yhat = model(x).argmax().item()
print(f"true: {class_name(y_true)} ({y_true})   pred: {class_name(yhat)} ({yhat})   P(true) = {base_conf:.4f}")
print(f"class {LEMON} = {class_name(LEMON)}")
plt.figure(figsize=(3, 3)); plt.imshow(denorm(x[0])); plt.axis('off');

### Resolve each parameter (grad ranking at the trained weights)

In [ ]:
layers = weighted_layers(model)

for k, s in enumerate(SWEEPS, start=1):
    name, layer = layers[s['li']]
    g = layer_grad(model, layer, x, y)
    score = g.abs()
    if s.get('out_idx') is not None:                       # only consider weights feeding one output unit / channel
        mask = torch.zeros_like(score).view(layer.weight.shape)
        mask[s['out_idx']] = 1
        score = score * mask.view(-1)
        assert TOP_N <= int(mask.sum()), "TOP_N larger than the number of weights feeding that output"
    _, idxs = score.topk(TOP_N)
    flat_idx = idxs[s['grad_index']].item()
    layer_num = s['li'] % len(layers) + 1          # 1-indexed for the theta superscript (fc of plain8 -> 8)
    coord = tuple(int(c) for c in np.unravel_index(flat_idx, layer.weight.shape))
    s.update(k=k, name=name, layer=layer, flat_idx=flat_idx, coord=coord, layer_num=layer_num,
             w0=layer.weight.view(-1)[flat_idx].item(), grad=g[flat_idx].item(),
             theta_label=theta_latex(coord, layer_num), out_dir=OUT_ROOT / f'p37_param_{k}')

print(f"{'k':>2} {'li':>3} {'layer':<16} {'L#':>3} {'grad_idx':>8} {'coord':<22} {'w0':>8} {'dL/dw':>8}  color")
for s in SWEEPS:
    print(f"{s['k']:>2} {s['li']:>3} {s['name']:<16} {s['layer_num']:>3} {s['grad_index']:>8} {str(s['coord']):<22} "
          f"{s['w0']:>8.4f} {s['grad']:>8.4f}  {s['color']}")
assert SWEEPS[0]['coord'][0] == LEMON, "final-layer parameter should sit in the lemon row of fc.weight"

### Sanity check: all five curves at the trained weights, dots at the probe values

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for s in SWEEPS:
    vals, res, _ = sweep_weight(model, s['layer'], s['flat_idx'], x, y, SPAN, N_POINTS)
    s['base_res'] = res
    ax.plot(vals, res, c=s['color'], linewidth=2, label=f"{s['k']}: {s['theta_label']}")
    _, probe_res, _ = sweep_weight(model, s['layer'], s['flat_idx'], x, y, SPAN, len(PROBE_VALS))
    ax.scatter(PROBE_VALS, probe_res, c=s['color'], s=20, zorder=10)
    ax.scatter(s['w0'], base_conf, c=s['color'], s=60, marker='*', zorder=11)
ax.set_xlim(-SPAN, SPAN); ax.set_ylim(*CURVE_YLIM); ax.grid(alpha=0.3); ax.legend(fontsize=9)
ax.set_xlabel(r'$\theta$'); ax.set_ylabel('P(correct)');
assert abs(correct_ans_conf(model, x, y) - base_conf) < 1e-6, "weights not restored!"

## Export

In [ ]:
def export_param(s):
    d = s['out_dir']
    dirs = {n: d / n for n in ['probs', 'equations', 'curve']}
    for p in dirs.values(): p.mkdir(parents=True, exist_ok=True)
    layer, flat_idx, w0, color = s['layer'], s['flat_idx'], s['w0'], s['color']

    # full-resolution curve (no dot)
    vals, res, _ = sweep_weight(model, layer, flat_idx, x, y, SPAN, N_POINTS)
    render_curve(vals, res, color, dirs['curve'] / 'curve.svg')

    probes = {}
    with torch.no_grad():
        w = layer.weight.view(-1)
        # trained value first, then the 11 probe values
        points = [('trained', w0)] + [(f'{i:02d}_{v:+.1f}', float(v)) for i, v in enumerate(PROBE_VALS)]
        for tag, v in tqdm(points, desc=f"param {s['k']} {s['theta_label']}"):
            w[flat_idx] = v
            probs = all_probs(model, x)
            p = float(probs[y_true])
            render_probs(probs, y_true, dirs['probs'] / f'{tag}.svg')
            render_equation(s['theta_label'], v, P_LABEL, p, color, dirs['equations'] / f'{tag}.svg')
            render_curve(vals, res, color, dirs['curve'] / f'curve_dot_{tag}.svg', dot=(v, p))
            probes[tag] = dict(theta=v, p_correct=p, pred=int(probs.argmax()), pred_class=class_name(int(probs.argmax())),
                               p_lemon=float(probs[LEMON]))
        w[flat_idx] = w0
    assert abs(correct_ans_conf(model, x, y) - base_conf) < 1e-6, "weights not restored!"

    json.dump({
        'param': s['k'], 'model': MODEL_NAME, 'img_idx': IMG_IDX, 'class_idx': y_true, 'class': class_name(y_true),
        'li': s['li'], 'layer_name': s['name'], 'layer_num': s['layer_num'], 'grad_index': s['grad_index'],
        'out_idx': s.get('out_idx'), 'flat_idx': s['flat_idx'], 'coord': list(s['coord']), 'w0': w0, 'grad': s['grad'],
        'color': color, 'theta_label': s['theta_label'], 'span': SPAN, 'n_points': N_POINTS, 'base_conf': base_conf,
        'probe_vals': [float(v) for v in PROBE_VALS], 'probes': probes,
        'curve': {'theta': vals.tolist(), 'p_correct': res.tolist()},
    }, open(d / 'summary.json', 'w'), indent=2)
    return vals, res, probes

In [ ]:
results = {}
for s in SWEEPS:
    if s['k'] in SWEEPS_TO_RUN:
        results[s['k']] = export_param(s)

### Preview: curve + probe points for each exported parameter

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(3.2 * len(results), 3), squeeze=False)
for ax, (k, (vals, res, probes)) in zip(axes[0], results.items()):
    s = SWEEPS[k - 1]
    ax.plot(vals, res, c=s['color'], linewidth=2)
    pv = [p for t, p in probes.items() if t != 'trained']
    ax.scatter([p['theta'] for p in pv], [p['p_correct'] for p in pv], c=s['color'], s=25, zorder=10)
    ax.scatter(probes['trained']['theta'], probes['trained']['p_correct'], c=s['color'], marker='*', s=80, zorder=11)
    ax.set_xlim(-SPAN, SPAN); ax.set_ylim(*CURVE_YLIM); ax.grid(alpha=0.3)
    ax.set_title(f"{k}: {s['theta_label']}", fontsize=9)
plt.tight_layout()

for k, (_, _, probes) in results.items():
    print(f"param {k}: " + "  ".join(f"{t}->{p['pred_class']}" for t, p in probes.items() if t != 'trained'))

### Which SVGs got written

In [ ]:
for s in SWEEPS:
    if s['k'] in results:
        for sub in ['probs', 'equations', 'curve']:
            files = sorted((s['out_dir'] / sub).glob('*.svg'))
            print(f"{s['out_dir'].name}/{sub}: {len(files)} svgs   e.g. {files[0].name} … {files[-1].name}")